# Cell Trajectory Divergence Analysis

This notebook implements analysis for detecting when STOP trial neural trajectories diverge from GO baseline.

Here we look at singel cell and check for their seperate divergence

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.special import gammaln
import sys

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

import importlib
import session_class
import cell_analysis
import multi_session_pca
importlib.reload(session_class)
importlib.reload(cell_analysis)
importlib.reload(multi_session_pca)

from session_class import Session
from cell_analysis import Cell
from multi_session_pca import MultiSessionPCA

import holoviews as hv
from holoviews import opts
from bokeh.io import output_notebook
output_notebook()
hv.extension('bokeh')

print("Imports loaded successfully!")

## 1. Load Data, instantiate session and populate variables 

### 1.A Variables and Parameters

In [ ]:
base_path = Path.cwd().parents[1] / 'data' / 'unified_cell_trial_data'
pickle_file = base_path / 'msn_fiona_cell_trial_data.pkl'

session_id = 'fi211025a'  

dir_map = {0: 'Right', 180: 'Left'}

SSD_NUM = 2
EPOK = [-50, 300]
BIN_SIZE = 50  # Bin width in ms
BIN_STEP = 10   # Bin step size in ms (sliding windows)
FR_MIN = 1.0    # Hz
DIRECTION = 180  # Left (has more STOP trials)

### 1.B get instances

In [ ]:
# Load MSN cell database
cell_df = pd.read_pickle(pickle_file)

# Select session 
session_data = cell_df[cell_df['trial_session'] == session_id]

# Create Session object
session = Session(session_data, verbose=True)

# Drop cells with missing trial type or direction data
session.drop_cells_with_missing_trial_type_or_dir_data()

print(f"\nSession {session_id} loaded successfully!")
print(f"Number of cells after filtering: {session.n_cells}")
print(f"Total cell-trial combinations: {len(session.data)}")

### 1.C Clean data
Remove GO trials with reaction time greater than mean SSD length (pluse maybe a buffer)

In [ ]:
session.data.columns

In [ ]:
def is_go_trial_with_valid_saccade(row, mean_ssd):
    if (row['type'] != 'GO'):
        return True
    if np.isnan(row['first_relevant_saccade']).any():
        return False
    saccade_onset = row['first_relevant_saccade'][0]
    return (saccade_onset - row['go_cue']) > mean_ssd

oring_data = session.data.copy()
print(oring_data.shape)

stop_trials = session.data[
    (session.data['type'] == 'STOP') & 
    (session.data['trial_failed'] == False) & 
    (session.data['dir'] == DIRECTION) &
    (session.data['ssd_number'] == SSD_NUM)
]
mean_ssd = stop_trials['ssd_len'].mean()
print(f"Mean SSD length for successful STOP SSD{SSD_NUM} trials: {mean_ssd:.2f} ms")

session.data[
    ~session.data.apply(lambda row: is_go_trial_with_valid_saccade(row, mean_ssd), axis=1)
]["trial_number"].value_counts().shape

# Filter GO trials
session.data = session.data[
    session.data.apply(lambda row: is_go_trial_with_valid_saccade(row, mean_ssd), axis=1)
]

print(f"Number of removed trials after GO trial filtering: {oring_data.shape[0] - session.data.shape[0]}")

## 2. Check Trial Counts

Verify we have enough GO and STOP SSD2 trials for each direction.

In [ ]:
# Count successful trials for GO and STOP SSD2 by direction

results = []
for dir in [0, 180]:
    dir_name = dir_map[dir]
    
    # GO trials (successful only) - count unique trial numbers
    go_count = len(session.data[
        (session.data['type'] == 'GO') & 
        (session.data['dir'] == dir) &
        (session.data['trial_failed'] == False)
    ].drop_duplicates(subset=['trial_number']))
    results.append({'Direction': dir_name, 'Type': 'GO', 'Trial Count': go_count})
    
    # STOP SSD2 trials (successful only)
    stop_count = len(session.data[
        (session.data['type'] == 'STOP') & 
        (session.data['dir'] == dir) & 
        (session.data['ssd_number'] == SSD_NUM) &
        (session.data['trial_failed'] == False)
    ].drop_duplicates(subset=['trial_number']))
    results.append({'Direction': dir_name, 'Type': 'STOP SSD2', 'Trial Count': stop_count})

results_df = pd.DataFrame(results)
print("\nSuccessful Trial Counts:")
print("=" * 40)
print(results_df.to_string(index=False))

In [ ]:
session.data['first_relevant_saccade']

## 3. Create Trial Number Lists

Extract unique trial numbers for GO and STOP trials by direction.

In [ ]:
def filter_trials(session_data, trial_type, direction, ssd_number=None, success_only=True):
    """
    Filter trials based on specified conditions.
    
    Parameters:
    -----------
    session_data : DataFrame
        Session data containing trial information
    trial_type : str
        'GO', 'STOP', or 'CONT'
    direction : int
        0 (right) or 180 (left)
    ssd_number : float, optional
        SSD level (1.0-4.0) for STOP/CONT trials
    success_only : bool
        If True, only include trials where trial_failed == False
    
    Returns:
    --------
    DataFrame : Filtered session data
    """
    # Base filter
    mask = (session_data['type'] == trial_type) & (session_data['dir'] == direction)
    
    # Add success filter if requested
    if success_only:
        mask &= (session_data['trial_failed'] == False)
    
    # Add SSD filter if specified
    if ssd_number is not None:
        mask &= (session_data['ssd_number'] == ssd_number)
    
    return session_data[mask]


def get_trial_numbers(session_data, trial_type, direction, ssd_number=None, success_only=True):
    """
    Extract unique trial numbers for specified conditions.
    
    Parameters:
    -----------
    session_data : DataFrame
        Session data containing trial information
    trial_type : str
        'GO', 'STOP', or 'CONT'
    direction : int
        0 (right) or 180 (left)
    ssd_number : float, optional
        SSD level (1.0-4.0) for STOP/CONT trials
    success_only : bool
        If True, only include trials where trial_failed == False
    
    Returns:
    --------
    list : Sorted list of unique trial numbers
    """
    filtered_data = filter_trials(session_data, trial_type, direction, ssd_number, success_only)
    return sorted(filtered_data['trial_number'].unique())


# Create trial lists for GO and STOP trials
trial_lists = {}

for dir, dir_name in dir_map.items():
    # GO trials
    trial_lists[f'go_{dir_name.lower()}'] = get_trial_numbers(
        session.data, 'GO', dir
    )
    
    # STOP trials with specified SSD
    trial_lists[f'stop_{dir_name.lower()}'] = get_trial_numbers(
        session.data, 'STOP', dir, ssd_number=SSD_NUM
    )

# Extract to individual variables for convenience
go_right_trials = trial_lists['go_right']
go_left_trials = trial_lists['go_left']
stop_right_trials = trial_lists['stop_right']
stop_left_trials = trial_lists['stop_left']

# Display results
print("Trial Number Lists:")
print("=" * 60)
print(f"\nGO Right (n={len(go_right_trials)}):")
print(f"\nGO Left (n={len(go_left_trials)}):")
print(f"\nSTOP SSD{SSD_NUM} Right (n={len(stop_right_trials)}):")
print(f"\nSTOP SSD{SSD_NUM} Left (n={len(stop_left_trials)}):")

## 4. Neuron Distribution per Trial

Analyze how many neurons are recorded for each trial.

In [ ]:
# Parameters for analysis
analysis_trial_type = 'STOP'
analysis_direction = DIRECTION
analysis_ssd_num = SSD_NUM

# Filter trials using the filter function
filtered_data = filter_trials(
    session.data, 
    trial_type=analysis_trial_type, 
    direction=analysis_direction, 
    ssd_number=analysis_ssd_num,
    success_only=True
)

# Count unique cells per trial
neurons_per_trial = filtered_data.groupby('trial_number')['cell_ID'].nunique().sort_index()

# Summary statistics
dir_name = dir_map[analysis_direction]
print(f"{dir_name} {analysis_trial_type} SSD{analysis_ssd_num} Trials - Neurons per Trial Distribution:")
print("=" * 60)
print(f"Total trials: {len(neurons_per_trial)}")
print(f"Mean neurons per trial: {neurons_per_trial.mean():.2f}")
print(f"Median neurons per trial: {neurons_per_trial.median():.0f}")
print(f"Min neurons per trial: {neurons_per_trial.min()}")
print(f"Max neurons per trial: {neurons_per_trial.max()}")
print(f"\nDistribution:")
print(neurons_per_trial.value_counts().sort_index())

# Visualize distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
ax1.hist(neurons_per_trial, bins=range(neurons_per_trial.min(), neurons_per_trial.max() + 2), 
         edgecolor='black', alpha=0.7)
ax1.set_xlabel('Number of Neurons')
ax1.set_ylabel('Number of Trials')
ax1.set_title(f'{dir_name} {analysis_trial_type} SSD{analysis_ssd_num} Trials: Neurons per Trial')
ax1.grid(axis='y', alpha=0.3)

# Per-trial view
ax2.bar(range(len(neurons_per_trial)), neurons_per_trial.values, edgecolor='black', alpha=0.7)
ax2.set_xlabel('Trial Index')
ax2.set_ylabel('Number of Neurons')
ax2.set_title(f'Neurons Recorded per Trial (n={len(neurons_per_trial)} trials)')
ax2.axhline(y=neurons_per_trial.mean(), color='r', linestyle='--', label=f'Mean: {neurons_per_trial.mean():.1f}')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Show trial-by-trial details
print(f"\n\nTrial-by-trial breakdown:")
print("=" * 60)
for trial_num, n_neurons in neurons_per_trial.items():
    print(f"Trial {trial_num}: {n_neurons} neurons")

## 5. Filter Neurons by STOP Trial Count

For the random sampling approach, we need neurons with sufficient STOP trials.
Filter to keep only neurons with at least 5 STOP trials in the specified direction.

In [ ]:
# Minimum STOP trials required per neuron
MIN_STOP_TRIALS = 8

# Filter STOP trials for the specified direction and SSD
stop_trial_data = filter_trials(
    session.data,
    trial_type='STOP',
    direction=DIRECTION,
    ssd_number=SSD_NUM,
    success_only=True
)

# Count STOP trials per neuron
stop_trials_per_neuron = stop_trial_data.groupby('cell_ID')['trial_number'].nunique()

# Filter neurons with sufficient STOP trials
valid_neurons = stop_trials_per_neuron[stop_trials_per_neuron >= MIN_STOP_TRIALS].index.tolist()

print(f"Neuron Filtering Results (Direction: {dir_map[DIRECTION]}, SSD{SSD_NUM}):")
print("=" * 60)
print(f"Total neurons in session: {session.n_cells}")
print(f"Neurons with >= {MIN_STOP_TRIALS} STOP trials: {len(valid_neurons)}")
print(f"Neurons filtered out: {session.n_cells - len(valid_neurons)}")
print(f"\nSTOP trials per neuron distribution:")
print(stop_trials_per_neuron.value_counts().sort_index())

# Display which neurons passed the filters
print(f"\n\nValid neuron IDs (n={len(valid_neurons)}):")
print(valid_neurons)

# Show statistics for valid neurons
if len(valid_neurons) > 0:
    valid_stop_counts = stop_trials_per_neuron[valid_neurons]
    print(f"\n\nValid neurons - STOP trial statistics:")
    print("=" * 60)
    print(f"Mean STOP trials: {valid_stop_counts.mean():.2f}")
    print(f"Median STOP trials: {valid_stop_counts.median():.0f}")
    print(f"Min STOP trials: {valid_stop_counts.min()}")
    print(f"Max STOP trials: {valid_stop_counts.max()}")

In [ ]:
# Also check GO trial counts for valid neurons
go_trial_data = filter_trials(
    session.data,
    trial_type='GO',
    direction=DIRECTION,
    success_only=True
)

# Count GO trials for valid neurons only
go_trials_per_neuron = go_trial_data.groupby('cell_ID')['trial_number'].nunique()
valid_go_counts = go_trials_per_neuron[valid_neurons]

print(f"GO trial counts for valid neurons (Direction: {dir_map[DIRECTION]}):")
print("=" * 60)
print(f"Mean GO trials: {valid_go_counts.mean():.2f}")
print(f"Median GO trials: {valid_go_counts.median():.0f}")
print(f"Min GO trials: {valid_go_counts.min()}")
print(f"Max GO trials: {valid_go_counts.max()}")

# Create summary dataframe
neuron_trial_summary = pd.DataFrame({
    'cell_ID': valid_neurons,
    'n_GO_trials': [go_trials_per_neuron.get(cell_id, 0) for cell_id in valid_neurons],
    'n_STOP_trials': [stop_trials_per_neuron.get(cell_id, 0) for cell_id in valid_neurons]
})

print(f"\n\nPer-neuron trial counts:")
print("=" * 60)
print(neuron_trial_summary.to_string(index=False))